# Baseline comparison with benchmark-consistent aggregation

The model selection:

- the GNN is evaluated only in the **Environment + Phylogeny** configuration;
- non-GNN baselines are retained only in the configuration with neither environmental nor phylogenetic inputs, as for their original implementations;

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = next(
    (parent for parent in (Path.cwd(), *Path.cwd().parents) if (parent / "dashboard.py").is_file()),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(f"Could not locate dashboard.py from {Path.cwd()}")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dashboard import (
    DEFAULT_RESULTS_DIR,
    load_metric_rows,
    aggregate_seed_level,
    aggregate_across_seeds,
)

trait_order = ['Alt', 'Ferstipe', 'FertileFrond', 'Spore', 'SterileFrond', 'SterileStipe']

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

print("Project root:", PROJECT_ROOT)
print("Default results directory:", DEFAULT_RESULTS_DIR)

2026-10-13 12:49:26.383 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-10-13 12:49:26.384 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-10-13 12:49:26.384 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager


Project root: /home/smonaco/newzeland-plants
Default results directory: /home/smonaco/newzeland-plants/benchmark_results


In [2]:
results_path = Path(str(DEFAULT_RESULTS_DIR)).expanduser()
traits_path = results_path.parent / "data" / "Ferns" / "FernMinMax.xlsx"

metric_rows = load_metric_rows(results_path, traits_path)
metric_rows.display_model = metric_rows.display_model.apply(lambda m: m.replace(" (prot1)", "") if isinstance(m, str) else m)

if metric_rows.empty:
    raise RuntimeError(f"No benchmark metric rows were found in {results_path}")

print(f"Loaded {len(metric_rows):,} fold-level metric rows.")
display(metric_rows.head())

Loaded 102,000 fold-level metric rows.


,trait,variable,n,RMSE,MAE,Pearson_r,Spearman_rho,fold,model,source,seed,measured_k_parameters,configuration_id,configuration,has_environment,has_phylogeny,experiment_dir,display_model,observed_n,trait_range,trait_iqr,NRMSE_IQR,NMAE_IQR,NRMSE_range,NMAE_range,k_parameters
80888,SterileStipe,min,32,63.430882,36.859375,0.359815,0.718614,fold_0,PHYLO_NN_prot1,r_baseline,46.0,NaN,baseline,Nessun input accessorio,False,False,no_env_no_phylo,Phylo Nn,165,1000.0,50.00,1.268618,0.737187,0.063431,0.036859,165.0
21116,Alt,max,34,418.689148,343.276184,NaN,NaN,fold_4,TRAINING_MEAN_prot1,r_baseline,103.0,NaN,baseline,Nessun input accessorio,False,False,no_env_no_phylo,Training Mean,167,2130.0,565.00,0.741043,0.607568,0.196568,0.161163,1.0
40484,FertileFrond,max,34,548.641724,335.773529,0.850337,0.775755,fold_3,PHYLO_KNN_prot1,r_baseline,117.0,NaN,baseline,Nessun input accessorio,False,False,no_env_no_phylo,Phylo Knn,170,4998.0,913.75,0.600429,0.367468,0.109772,0.067182,34.0
21112,Alt,min,34,48.472031,32.432552,NaN,NaN,fold_4,TRAINING_MEAN_prot1,r_baseline,103.0,NaN,baseline,Nessun input accessorio,False,False,no_env_no_phylo,Training Mean,167,900.0,10.00,4.847203,3.243255,0.053858,0.036036,1.0
21108,Spore,max,29,31033.775391,6376.288086,NaN,NaN,fold_4,TRAINING_MEAN_prot1,r_baseline,103.0,NaN,baseline,Nessun input accessorio,False,False,no_env_no_phylo,Training Mean,158,168123.0,823.25,37.696660,7.745263,0.184590,0.037926,1.0


In [3]:
is_gnn = metric_rows["model"].eq("gnn")

keep_full_gnn = (
    is_gnn
    & metric_rows["has_environment"].astype(bool)
    & metric_rows["has_phylogeny"].astype(bool)
)

keep_baselines = (
    ~is_gnn
    & ~metric_rows["has_environment"].astype(bool)
    & ~metric_rows["has_phylogeny"].astype(bool)
)

filtered_metric_rows = metric_rows.loc[keep_full_gnn | keep_baselines].copy()

print("Unique fold-level seeds per model before seed aggregation:")
display(
    filtered_metric_rows
    .groupby("display_model", as_index=False)
    .agg(
        n_seed_ids=("seed", "nunique"),
        n_fold_rows=("seed", "size"),
    )
    .sort_values("display_model")
)

Unique fold-level seeds per model before seed aggregation:


,display_model,n_seed_ids,n_fold_rows
0,GNN,60,3600
1,Mice,60,3600
2,Phylo Knn,61,3660
3,Phylo Nn,61,3660
4,Rphylopars Bm,61,3660
5,Training Mean,61,3660
6,Training Median,61,3660


## 3. Aggregate folds into one value per seed

We apply the benchmark aggregation rules:

- observation-weighted RMSE;
- observation-weighted MAE;
- weighted Fisher-z mean for correlations;
- normalized errors recomputed from the aggregated RMSE/MAE and the original trait scale.

The resulting table contains one row per:

**model × configuration × seed × trait × output variable**.

In [4]:
seed_level = aggregate_seed_level(filtered_metric_rows)

if seed_level.empty:
    raise RuntimeError("Seed-level aggregation returned no rows.")

seed_level = seed_level.rename(
    columns={
        "trait": "Trait",
        "variable": "Variable",
        "display_model": "Model",
    }
)

print(f"Seed-level rows: {len(seed_level):,}")

seed_counts = (
    seed_level.groupby("Model", as_index=False)
    .agg(
        Seeds=("seed", "nunique"),
        Trait_outputs=("Trait", lambda s: seed_level.loc[s.index, ["Trait", "Variable"]].drop_duplicates().shape[0]),
    )
)

display(seed_counts.sort_values("Model"))
display(seed_level.head())

Seed-level rows: 5,100


,Model,Seeds,Trait_outputs
0,GNN,60,12
1,Mice,60,12
2,Phylo Knn,61,12
3,Phylo Nn,61,12
4,Rphylopars Bm,61,12
5,Training Mean,61,12
6,Training Median,61,12


,configuration_id,configuration,has_environment,has_phylogeny,model,source,Model,seed,Trait,Variable,n,RMSE,MAE,Pearson_r,Spearman_rho,folds,trait_iqr,trait_range,observed_n,k_parameters,NRMSE_IQR,NRMSE_range,NMAE_IQR,AIC
0,baseline,Nessun input accessorio,False,False,PHYLO_NN_prot1,r_baseline,Phylo Nn,46.0,SterileStipe,min,161.0,92.279263,36.832298,0.317328,0.595087,5,50.00,1000.0,165,165.0,1.845585,0.092279,0.736646,1786.991861
1,baseline,Nessun input accessorio,False,False,TRAINING_MEAN_prot1,r_baseline,Training Mean,103.0,Alt,max,167.0,445.856235,356.211779,NaN,NaN,5,565.00,2130.0,167,1.0,0.789126,0.209322,0.630463,2039.398850
2,baseline,Nessun input accessorio,False,False,PHYLO_KNN_prot1,r_baseline,Phylo Knn,117.0,FertileFrond,max,166.0,585.309792,369.261437,0.733685,0.706192,5,913.75,4998.0,170,34.0,0.640558,0.117109,0.404116,2183.550900
3,baseline,Nessun input accessorio,False,False,TRAINING_MEAN_prot1,r_baseline,Training Mean,103.0,Alt,min,167.0,99.930624,39.546008,NaN,NaN,5,10.00,900.0,167,1.0,9.993062,0.111034,3.954601,1539.895044
4,baseline,Nessun input accessorio,False,False,TRAINING_MEAN_prot1,r_baseline,Training Mean,103.0,Spore,max,157.0,13421.625373,2419.570410,NaN,NaN,5,823.25,168123.0,158,1.0,16.303219,0.079832,2.939047,2986.451471


## 4. Aggregate repeated seeds

`aggregate_across_seeds()` reproduces the benchmark summary exactly:

- arithmetic mean across independent seed-level runs;
- sample standard deviation across seeds;
- number of seeds retained in `n_seeds`.

In [5]:
seed_level_for_dashboard = seed_level.rename(
    columns={
        "Trait": "trait",
        "Variable": "variable",
        "Model": "display_model",
    }
)

summary = aggregate_across_seeds(seed_level_for_dashboard)
summary = summary[~summary.source.eq("deterministic_baseline")]

summary = summary.rename(
    columns={
        "trait": "Trait",
        "variable": "Variable",
        "display_model": "Model",
        "n_seeds": "Seeds",
    }
)

print(f"Across-seed summary rows: {len(summary):,}")
display(
    summary[
        [
            "Model", "Trait", "Variable", "Seeds",
            "NRMSE_IQR", "NRMSE_IQR_std",
            "RMSE", "RMSE_std",
            "Pearson_r", "Pearson_r_std",
        ]
    ].head(20)
)

Across-seed summary rows: 84


,Model,Trait,Variable,Seeds,NRMSE_IQR,NRMSE_IQR_std,RMSE,RMSE_std,Pearson_r,Pearson_r_std
0,Phylo Nn,SterileStipe,min,60,1.862143,0.044015,93.107162,2.200734,0.316324,0.061150
1,Training Mean,Alt,max,60,0.790115,0.003108,446.414860,1.756223,NaN,0.000000
2,Phylo Knn,FertileFrond,max,60,0.655967,0.021408,599.389926,19.561540,0.764199,0.021822
3,Training Mean,Alt,min,60,9.993644,0.038805,99.936438,0.388053,NaN,0.000000
4,Training Mean,Spore,max,60,16.312062,0.009467,13428.905182,7.793968,NaN,0.000000
5,Phylo Knn,SterileFrond,min,60,1.751516,0.068865,294.254654,11.569367,0.792924,0.047317
6,Training Mean,Spore,min,60,18.571059,0.009460,7156.047883,3.645206,NaN,0.000000
7,Phylo Knn,SterileFrond,max,60,0.657559,0.021603,600.844843,19.739298,0.759254,0.021487
8,Phylo Knn,FertileFrond,min,60,1.748490,0.069017,293.746287,11.594777,0.795255,0.046788
9,Phylo Nn,SterileStipe,max,60,0.633705,0.027917,297.841170,13.121107,0.634026,0.038522


## 5. NRMSE-IQR comparison table

The table below reproduces the original analysis, but uses the correctly aggregated **mean NRMSE-IQR across seeds**.

Lower values are better.

- **bold**: best model for the trait/output column;
- *underlined*: second-best model.

In [6]:
MODEL_ORDER = [
    "Training Mean",
    "Training Median",
    "Phylo Nn",
    "Phylo Knn",
    "Mice",
    "Rphylopars Bm",
    "GNN",
]

METRIC = "NRMSE_IQR"

comparison_table = (
    summary
    .pivot(index="Model", columns=["Trait", "Variable"], values=METRIC)
)

extra_models = [m for m in comparison_table.index if m not in MODEL_ORDER]
comparison_table = comparison_table.reindex(MODEL_ORDER + extra_models)

display(comparison_table)

Trait           SterileStipe       Alt FertileFrond        Alt      Spore SterileFrond      Spore SterileFrond FertileFrond SterileStipe  Ferstipe          
Variable                 min       max          max        min        max          min        min          max          min          max       max       min
Model                                                                                                                                                       
Training Mean       1.821806  0.790115     0.936154   9.993644  16.312062     2.449897  18.571059     0.931088     2.448753     0.749561  0.736030  1.753885
Training Median     1.867099  0.786641     0.968974  10.287848  16.271437     2.535766  18.526595     0.960652     2.529349     0.792943  0.774923  1.798247
Phylo Nn            1.862143  0.925023     0.643155  12.517999  16.489511     1.818472  18.777480     0.659835     1.812684     0.633705  0.605727  1.802716
Phylo Knn           1.835270  0.806965     0.655967  11.162848  16.548804     1.751516  18.831188     0.657559     1.748490     0.562181  0.549385  1.768512
Mice                1.991034  0.986360     1.331082  10.283699  16.392769     2.723219  25.542617     1.326539     2.723900     0.322663  0.310842  1.823951
Rphylopars Bm       1.800659  0.804687     0.573897  10.940228  16.471699     1.609122  18.774824     0.590790     1.604869     0.584336  0.560022  1.736720
GNN                 1.551305  0.797260     0.530430  10.262097  16.261663     1.722908  18.534105     0.532841     1.662412     0.473150  0.453872  1.518476

In [7]:
import numpy as np

def highlight_top_two(column, method="min", ascending=True, kind="style"):
    """
    Highlight the best and second-best values in a column.

    Supports both:
    - numeric values
    - strings formatted as "mean ± SD"

    Ranking is based on the numeric value itself, or on the mean when the
    input is formatted as "mean ± SD".
    """

    def extract_value(x):
        if pd.isna(x):
            return np.nan

        # Already numeric
        if isinstance(x, (int, float, np.number)):
            return float(x)

        # Handle strings such as "1.234 ± 0.056"
        text = str(x).strip()

        if "±" in text:
            text = text.split("±", 1)[0].strip()

        return pd.to_numeric(text, errors="coerce")

    numeric = column.map(extract_value)

    ranks = numeric.rank(method=method, ascending=ascending, na_option="bottom",
    )

    if kind == "style":
        return [
            "background-color: lightgreen; font-weight: bold"
            if rank == 1
            else "background-color: khaki; text-decoration: underline"
            if rank == 2
            else ""
            for rank in ranks
        ]

    if kind == "latex":
        return [
            "bfseries: ;"
            if rank == 1
            else "itshape: ; underline: --rwrap;"
            if rank == 2
            else ""
            for rank in ranks
        ]

    raise ValueError("kind must be either 'style' or 'latex'")



styled_table = (
    comparison_table.style
    .apply(
        highlight_top_two,
        axis=0,
        method="min",
        ascending=True,
        kind="style",
    )
    .format(precision=3, na_rep="--")
)

display(styled_table)

## 6. Seed counts accompanying the performance table

A mean from one deterministic run and a mean from many stochastic runs do not have the same uncertainty.

This companion table makes the number of independent runs explicit.

In [8]:
seed_count_table = (
    summary
    .pivot(index="Model", columns=["Trait", "Variable"], values="Seeds")
    .reindex(comparison_table.index)
)

display(seed_count_table.style.format(precision=0, na_rep="--"))

## 7. Optional table with mean ± standard deviation

For stochastic models, this table exposes between-seed variability directly.

Each cell is formatted as:

$$\mathrm{mean}\ \pm\ \mathrm{SD}.$$

For deterministic baselines the standard deviation is zero by construction.

In [9]:

mean_table = (
    summary
    .pivot(index="Model", columns=["Trait", "Variable"], values="NRMSE_IQR")
    .reindex(comparison_table.index)
)

std_table = (
    summary
    .pivot(index="Model", columns=["Trait", "Variable"], values="NRMSE_IQR_std")
    .reindex(comparison_table.index)
)

ordered_columns = pd.MultiIndex.from_product(
        [trait_order, ["min", "max"]],
        names=["Trait", "Variable"]
    ) 

mean_sd_table = mean_table.copy().astype(object)

for row in mean_sd_table.index:
    for col in mean_sd_table.columns:
        mean_value = mean_table.loc[row, col]
        std_value = std_table.loc[row, col]

        if pd.isna(mean_value):
            mean_sd_table.loc[row, col] = "--"
        elif pd.isna(std_value):
            mean_sd_table.loc[row, col] = f"{mean_value:.3f}"
        else:
            mean_sd_table.loc[row, col] = f"{mean_value:.3f} ± {std_value:.3f}"

mean_sd_table.rename(index={m: m.replace(" (prot1)", "") for m in mean_sd_table.index}, inplace=True)
mean_table.rename(index={m: m.replace(" (prot1)", "") for m in mean_table.index}, inplace=True)

# check if columns are present before reindexing to avoid errors
assert set(ordered_columns) >= set(mean_sd_table.columns), "Columns of mean_sd_table do not match the expected order."
mean_sd_table = mean_sd_table.reindex(ordered_columns, axis=1)
mean_table = mean_table.reindex(ordered_columns, axis=1)

mean_sd_table.style.apply(
    highlight_top_two,
    axis=0,
    ascending=True,
    kind="style",
)


## 8. LaTeX export

This reproduces the publication-oriented table with the same ranking rule.

The ranking is applied to the correctly aggregated mean NRMSE-IQR values.

In [10]:
styled_result_latex = (
    mean_table.style
    .apply(
        highlight_top_two,
        axis=0,
        method="min",
        ascending=True,
        kind="latex",
    )
    .format(precision=3, na_rep="--")
    .format_index(escape="latex", axis=0)
    .format_index(escape="latex", axis=1)
)

latex = styled_result_latex.to_latex(
    caption=(
        "Performance in terms of mean NRMSE-IQR across independent benchmark seeds "
        "for morphological traits and their output components. Fold-level metrics "
        "are first aggregated within each seed using observation-weighted benchmark "
        "rules. The best (lowest) value for each output variable is shown in bold, "
        "and the second-best value is underlined."
    ),
    label="tab:performance_traits",
    hrules=True,
    multicol_align="c",
    column_format="l" + "c" * mean_sd_table.shape[1],
)

print(latex)

\begin{table}
\caption{Performance in terms of mean NRMSE-IQR across independent benchmark seeds for morphological traits and their output components. Fold-level metrics are first aggregated within each seed using observation-weighted benchmark rules. The best (lowest) value for each output variable is shown in bold, and the second-best value is underlined.}
\label{tab:performance_traits}
\begin{tabular}{lcccccccccccc}
\toprule
Trait & \multicolumn{2}{c}{Alt} & \multicolumn{2}{c}{Ferstipe} & \multicolumn{2}{c}{FertileFrond} & \multicolumn{2}{c}{Spore} & \multicolumn{2}{c}{SterileFrond} & \multicolumn{2}{c}{SterileStipe} \\
Variable & min & max & min & max & min & max & min & max & min & max & min & max \\
Model &  &  &  &  &  &  &  &  &  &  &  &  \\
\midrule
Training Mean & \bfseries 9.994 & \itshape \underline{0.790} & 1.754 & 0.736 & 2.449 & 0.936 & 18.571 & 16.312 & 2.450 & 0.931 & 1.822 & 0.750 \\
Training Median & 10.288 & \bfseries 0.787 & 1.798 & 0.775 & 2.529 & 0.969 & \bfserie

# Interpretation

The final performance table is now directly compatible with the aggregation used by the benchmark dashboard:

1. **folds are aggregated within each seed using the correct metric-specific rule;**
2. **independent seeds are then averaged;**
3. **between-seed variability is retained as a standard deviation;**
4. **deterministic baselines are not artificially replicated;**
5. **baseline rows duplicated across feature configurations are removed before aggregation.**

This makes the baseline comparison methodologically consistent with the benchmark and with seed-level ablation analyses.